# Ridge plot + cdf: LFC per nucleotide  
nucleotides classified with consequences and plotted with LFC per time unit
> /media/scratch/fy2306/projects/base_editing/script/grna_plot_lfc_seq_wt_duplicated.ipynb
> /media/scratch/fy2306/projects/base_editing/script/archive/grna_plot_cons_ridge.ipynb

In [ ]:
import pandas as pd
import numpy as np
from Bio import SeqIO
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import re
import pyranges as pr
import matplotlib as mpl
from matplotlib import font_manager

arial_path = "/media/scratch/fy2306/tools/fonts"
font_files = font_manager.findSystemFonts(fontpaths=arial_path)

for file in font_files:
    font_manager.fontManager.addfont(file)
    
mpl.rcParams['font.family'] = 'Arial'

In [ ]:
id_prefix_list = [
	"MYC_GFP_", "MYC_SNP1_", "MYC_SNP2_", "MYC_SNP3_", "MYC_STOP_"
]

In [ ]:
dict_cons_order = {
	"MYC-wt": [
		"negative_controls",
		"3_prime_UTR \n + downstream_flanking",
		"intron",
		"5_prime_UTR \n + upstream_flanking",
		"coding",
		"repeats"
		]
}

dict_cons_order_more = {
	"MYC-wt": [
		"negative_controls",
		"downstream_flanking",
		"3_prime_UTR", 
		"intron_2",
		"intron_1",
		"5_prime_UTR",
		"promoter",
		"upstream_flanking",
		"coding",
		"intron_splice"
		]
}

nc_seq_name = ["NT", "AAVS1", "Random"]

In [ ]:
def load_df(date, treatment_full, test_name, method_name, baseline, mode):
	if mode == "gene_summary":
		df_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{test_name}/{method_name}/myc_repeats/MYC_U1.{treatment_full}_vs_{baseline}.gene_summary.txt"
		df = pd.read_csv(df_path, sep="\t", usecols=["id", "neg|lfc"])
	elif mode == "sgrna_summary":
		df_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{test_name}/MYC_U1.{treatment_full}_vs_{baseline}.sgrna_summary.txt"
		df = pd.read_csv(df_path, sep="\t", usecols=["sgrna", "Gene", "LFC"])
		df = df[df["Gene"].isin(nc_seq_name)]
		df = df.drop("Gene", axis=1)

	return df

In [ ]:
# remove negative controls that can be mapped to the genome (with bowtie)
mapped_ctrl_path = "/media/scratch/fy2306/projects/base_editing/data/bowtie/all/all_sgrna_seqs.bowtie_hg38.processed.tsv"
mapped_ctrl_df = pd.read_csv(mapped_ctrl_path, sep="\t")
mapped_ctrl_df = mapped_ctrl_df[
    mapped_ctrl_df['id'].str.startswith(('Random', 'Non-targeting-controls', 'AAVS1'))
    & ~(mapped_ctrl_df['id'].str.startswith('AAVS1') & (mapped_ctrl_df['Alignments_NM0'] == 1) & (mapped_ctrl_df['Alignments_NM1'] == 0))
]
mapped_ctrl_list = mapped_ctrl_df["id"].tolist()
print(len(mapped_ctrl_list))
all_ctrl = [
    record.id for record in SeqIO.parse("/media/scratch/fy2306/projects/base_editing/data/bowtie/negative_controls/input/negative_controls.fa", "fasta")
    ]
clean_ctrl_list = list(set(all_ctrl)-set(mapped_ctrl_list))
print(len(clean_ctrl_list))
print(len(all_ctrl))

In [ ]:
# randomly group nc sgrnas
def group_and_aggregate(df, group_size, cons_col, random_state=None):
    df_shuffled = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    num_groups = len(df_shuffled) // group_size
    df_trimmed = df_shuffled.iloc[:num_groups * group_size]
    
    grouped = np.array_split(df_trimmed, num_groups)
    
    aggregated = []
    for group in grouped:
        sgrna_concat = ','.join(group['sgrna'].astype(str))
        avg_lfc = group['per time unit LFC'].mean()
        cons_val = group[cons_col].iloc[0]
        aggregated.append({
            'sgrna': sgrna_concat,
            'per time unit LFC': avg_lfc,
            cons_col: cons_val
        })
    
    return pd.DataFrame(aggregated)

In [ ]:
def ridge_plot_more(
		gene, myc_var, edited_pos_chr_col, treatment_col,
		date,
		test_name,
		nc_test_name,
		merge_method_nucleotide, 
		win_size,
		baseline,
		treatments,
		day_numbers,
		merge_method_replicate, per_time_unit=True):
	
	# process the cutting site
	df_myc_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{test_name}/{merge_method_nucleotide}/myc_repeats/{win_size}/MYC_U1.gene_summary.txt"
	df_myc = pd.read_csv(df_myc_path, sep="\t")
	pattern = f"^({'|'.join(map(re.escape, id_prefix_list))})([0-9]*\.?[0-9]+)"
	df_myc = df_myc[df_myc['id'].str.match(pattern)]
	df_myc[edited_pos_chr_col] = df_myc['id'].str.extract(pattern)[1].astype(int)

	# merge
	treatment_lfc_columns = [f"{treatment}_vs_{baseline}|neg|lfc" for treatment in treatments]

	if per_time_unit:
		for col, day in zip(treatment_lfc_columns, day_numbers):
			df_myc[col] = df_myc[col] / (day + 0) # per time unit. 

	if merge_method_replicate == "mean":
		df_myc["per time unit LFC"] = df_myc[treatment_lfc_columns].mean(axis=1)
	df_myc = df_myc[['id', 'per time unit LFC', edited_pos_chr_col]]
	# columns: id, per time unit LFC, edited_pos_chr_col

	# load in lib to add cons col
	cons_col = treatment_col + "_consequences_more"

	lib_path = f"/media/scratch/fy2306/projects/base_editing/data/grna_type/MYC-lib-for-mageck.type.{myc_var}.organized.txt"
	df_lib = pd.read_csv(lib_path, sep="\t", usecols=[edited_pos_chr_col, cons_col])
	df_lib = df_lib.drop_duplicates(subset=edited_pos_chr_col)
	df_myc = pd.merge(df_myc, df_lib, on=edited_pos_chr_col, how='left')
	print(df_myc.duplicated().sum())

	# add repeats
	rmsk_path = "/media/dna/fy2306/genomes/hg38/RepeatMasker/rmsk.txt"
	rmsk_colnames = ["bin", "swScore", "milliDiv", "milliDel", "milliIns", "genoName", "genoStart", "genoEnd", "genoLeft", "strand", "repName", "repClass", "repFamily", "repStart", "repEnd", "repLeft", "id"]
	rmsk = pd.read_csv(rmsk_path, sep="\t", header=None, names=rmsk_colnames)
	rmsk = rmsk[rmsk['genoName'] == "chr8"]
	# rmsk: 0-based
	rmsk_ranges = pr.PyRanges(rmsk.rename(columns={
	'genoName': 'Chromosome',
	'genoStart': 'Start',
	'genoEnd': 'End'
	}))

	# base_pos: 1-based
	df_myc['Start'] = df_myc[edited_pos_chr_col] - 1
	df_myc['End'] = df_myc['Start'] + 1
	df_myc['Chromosome'] = 'chr8'

	df_myc["orig_index"] = df_myc.index
	df_ranges = pr.PyRanges(df_myc[['Chromosome', 'Start', 'End', 'orig_index']])

	overlap = df_ranges.intersect(rmsk_ranges, how="containment")	# in pyranges: “containment” reports intervals where the overlapping is contained within it, gives you the intervals in self be completely within the intervals in other
	overlap_indices = overlap.df['orig_index'].unique()

	df_myc["indel_repeats"] = False
	df_myc.loc[overlap_indices, "indel_repeats"] = True

	# degenerate cons
	df_myc.loc[df_myc["indel_repeats"] == True, cons_col] = 'repeats'
	df_myc = df_myc[df_myc["indel_repeats"] == False]	# remove repeats
	df_myc.loc[
		(df_myc[edited_pos_chr_col] >= (127736084-1000)) & (df_myc[cons_col] == 'upstream_flanking'),
		cons_col
		] = 'promoter'

	df_myc = df_myc.drop(edited_pos_chr_col, axis=1)
	# columns: id, per time unit LFC, cons_col

	# process the negative controls
	df_nc = None
	for treatment in treatments:
		df_nc_path = f"/media/scratch/fy2306/projects/base_editing/data/{date}/mageck/{nc_test_name}/MYC_U1.{treatment}_vs_{baseline}.sgrna_summary.txt"
		df_nc_rep = pd.read_csv(df_nc_path, sep="\t", usecols=["sgrna", "Gene", "LFC"])
		df_nc_rep = df_nc_rep[df_nc_rep["Gene"].isin(["AAVS1", "NT", "Random"])]
		df_nc_rep = df_nc_rep.drop('Gene', axis=1)
		df_nc_rep = df_nc_rep.rename(columns={"LFC": f"LFC_{treatment}"})
		if df_nc is None:
			df_nc = df_nc_rep
		else:
			# Merge on 'sgrna'
			df_nc = pd.merge(df_nc, df_nc_rep, on="sgrna", how="inner")
	print(len(df_nc))

	treatment_lfc_nc_columns = [f"LFC_{treatment}" for treatment in treatments]

	if per_time_unit:
		for col, day in zip(treatment_lfc_nc_columns, day_numbers):
			df_nc[col] = df_nc[col] / (day + 0) # per time unit.

	if merge_method_replicate == "mean":
		df_nc["per time unit LFC"] = df_nc[treatment_lfc_nc_columns].mean(axis=1)

	# remove mismatches
	df_nc = df_nc[~(df_nc["sgrna"].isin(mapped_ctrl_list))]
	df_nc[cons_col] = "negative_controls"
	df_nc = df_nc[['sgrna', 'per time unit LFC', cons_col]]
	print(f"negative control sgRNAs median: {df_nc['per time unit LFC'].median()}")
	print(f"negative control sgRNAs mean: {df_nc['per time unit LFC'].mean()}")

	# columns: sgrna, per time unit LFC, cons_col

	# random group of df_nc
	n_repeats = 1000
	all_results = []

	for seed in range(n_repeats):
		df_nc_result = group_and_aggregate(df_nc, group_size=2*int(win_size), cons_col=cons_col, random_state=seed)
		all_results.append(df_nc_result)

	df_nc_combined = pd.concat(all_results, ignore_index=True)
	nc_combined_median_value = df_nc_combined["per time unit LFC"].median()
	print(nc_combined_median_value)

	# concat
	df = pd.concat([df_myc, df_nc_combined], ignore_index=True)
	df["per time unit LFC"] = df["per time unit LFC"] - nc_combined_median_value
	cons_order = dict_cons_order_more.get(gene)
	print("Order from dict:", cons_order)
	df[cons_col] = pd.Categorical(df[cons_col], categories=cons_order, ordered=True)

	# stats
	stat_df_1 = df.groupby([cons_col]).size().reset_index(name='TotalCount')

	bottom_line = 0 # df_nc_combined['per time unit LFC'].quantile(0.5)
	stat_df_2 = df[df["per time unit LFC"] <= bottom_line].groupby([cons_col]).size().reset_index(name='BelowCount')

	stat_df = pd.merge(stat_df_1, stat_df_2, on=cons_col, how='left')
	stat_df["BelowCount"] = stat_df["BelowCount"].fillna(0).astype(int)
	stat_df["TotalCount"] = stat_df["TotalCount"].astype(int)
	stat_df["percent"] = (stat_df["BelowCount"] / stat_df["TotalCount"]) * 100
	stat_df["percent_rounded"] = stat_df["percent"].round(1)
	stat_df["text_bottom"] = stat_df["percent_rounded"].astype(str) + "% \n (" + stat_df["BelowCount"].astype(str) + " / " + stat_df["TotalCount"].astype(str) + ")"
	df = pd.merge(df, stat_df[[cons_col, 'text_bottom']], on=cons_col, how="left")
	print(df.duplicated().sum())

	first_val = "negative_controls"
	cons_order = [first_val] + (stat_df.sort_values("percent_rounded", ascending=False)
								.loc[lambda d: d[cons_col].ne(first_val), cons_col]
								.drop_duplicates()
								.tolist())

	# plot
	g = sns.FacetGrid(df, row=cons_col, hue=cons_col, aspect=6, height=1.1, row_order=cons_order)

	g.map(sns.kdeplot, 'per time unit LFC',
		clip=(-0.10, 0.05),
		fill=True, 
		alpha=1, 
		linewidth=1,
		color="#107F80")

	g.map(sns.kdeplot, 'per time unit LFC',
		clip_on=False, 
		clip=(-0.10, 0.05),
		color="w", 
		lw=2)
	
	for ax in g.axes.flat:
		ax.set_facecolor((0, 0, 0, 0))

	g.map(plt.axvline, x=bottom_line, color='black', linestyle='--', linewidth=1)

	def custom_label(x, color, label):
		ax = plt.gca()
		label_text = df[df[cons_col] == label].iloc[0]['text_bottom']
		ax.text(0, .3, f'{label} \n LFC < {str(f"{bottom_line:.0f}")}: {label_text}', fontweight="bold", color="black",
				ha="left", va="center", transform=ax.transAxes, fontsize=15)

	g.map(custom_label, "per time unit LFC")

	medians = df.groupby(cons_col)['per time unit LFC'].median()

	for ax, cons_val in zip(g.axes.flat, g.row_names):
		median_val = medians.loc[cons_val]
		ax.axvline(median_val, color='#FF0066', linestyle='-', linewidth=2, ymin=0, ymax=0.35)

	g.figure.subplots_adjust(hspace=-.4)

	g.set_titles("")
	g.set(yticks=[], ylabel="")
	g.despine(bottom=True, left=True)

	# plt.suptitle(f"per time unit LFC", fontsize=18, fontweight='bold')
	for i, ax in enumerate(g.axes.flat):
		if i == len(g.axes.flat) - 1:
			ax.set_xticks([-0.10, -0.05, 0, 0.05])
			ax.tick_params(axis='x', labelsize=14)
			ax.set_xlabel("base pair phenotype score", fontsize=16)
		else:
			ax.tick_params(axis='x', bottom=False, labelbottom=False)
	plt.savefig("/media/scratch/fy2306/projects/base_editing/plots/screening/basepair_lfc_category.detailed.pdf", 
				bbox_inches="tight",
				dpi=300,              
				transparent=True,
				format='pdf')
	plt.show()

In [ ]:
gene = "MYC-wt"
myc_var = "myc2_indel"
edited_pos_chr_col = "indel_base_pos_chr"
treatment_col = "indel"
date = "20250416"
test_name = "test-1-standard-nucleotide/test"
nc_test_name = "test-1-standard/test"
merge_method_nucleotide = "mean"
win_size = "4"
baseline = "Cas9-HMOI_D0"
treatments = ["Cas9-LMOI_D20", "Cas9-LMOI_D8", "Cas9_SpRY_L_MOI_D_20", "Cas9_SpRY_L_MOI_D_8"]
day_numbers = [20, 8, 20, 8]
merge_method_replicate = "mean"
ridge_plot_more(
		gene, myc_var, edited_pos_chr_col, treatment_col,
		date,
		test_name,
		nc_test_name,
		merge_method_nucleotide, 
		win_size,
		baseline,
		treatments,
		day_numbers,
		merge_method_replicate, per_time_unit=True)